In [ ]:
from langchain_community.chat_models import ChatTongyi
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.vectorstores import InMemoryVectorStore

model = ChatTongyi(model="qwen3-max")
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","以我提供的已知仓靠资料为主，简洁和专业的回答用户问题。参考资料:{content}"),
        ("user","用户提问:{input}")
    ]
)

vector_store=InMemoryVectorStore(embedding=DashScopeEmbeddings(model="text-embedding-v4"))

# 准备资料（向量库的资料）
# add_texts 传入一个list[str]
vector_store.add_texts(
    ["减肥就要少吃多练","减脂期间吃东西很重要，清淡少油控制卡路里摄入并运动起来","跑步是好运动"]
)

input_text="怎么减肥？"

# langchain中向量存储对象，有一个方法：as_retriever，可以返回一个Runnable接口的子类实例对象
retriever=vector_store.as_retriever(search_kwargs={"k":2})

def format_func(docs:list[Document]):
    if not docs:
        return "无相关参考资料"
    formatted_str="["
    for doc in docs:
        formatted_str+=doc.page_content
    formatted_str+="]"
    return formatted_str

# chain
chain=(
    {"intput":RunnablePassthrough(),"content":retriever|format_func} | prompt |
    model | StrOutputParser()
)

# RunnablePassthrough的作用是在invoke调用的时候截流也拿到input_text
res=chain.invoke(input_text)
"""
retriever:
    - 输入 ： 用户提问        str
    - 输出 ： 向量库检索结果   list[Document]
prompt:
    - 输入 ： 用户提问 + 向量库检索结果  dict
    - 输出 ： 完整的提示词             PromptValue
"""